# AI Resume Screening Assistant with Gemini

This notebook demonstrates how to build an AI-powered resume screening assistant using the Gemini 3.5 Flash model. The assistant takes a job description and a candidate's resume, then extracts key information, assesses fit, and provides a recommendation in a structured JSON format.

## Problem Statement


A company receives hundreds of resumes for different job openings. This assistant aims to automate the initial screening process by:

1.  Determining whether the candidate is a Strong Fit, Partial Fit, or Not a Fit
2.  Extracting the candidate's key skills
3.  Identifying years of experience
4.  Giving a short reason for the decision
5.  Suggesting whether the recruiter should shortlist the candidate
6.  Returning the result as structured JSON

## Setup and Imports

In [43]:
from google import genai
from google.genai import types
from google.colab import userdata
import json

In [44]:
# Load API Key from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

## Define Inputs: Job Description and Resume

In [45]:
job_description = """
We are looking for a Machine Learning Engineer.

Requirements:
- Python
- Machine Learning
- SQL
- 3+ years of experience
- Experience with NLP is preferred
"""

resume = """
Rahul has 4 years of experience in software engineering.
He has worked extensively with Python, SQL and machine learning.
He has built NLP classification models and recommendation systems.
He has also worked with AWS and Docker.
"""

## System Instruction for Gemini Model

In [46]:
# This instruction guides the Gemini model on how to process the inputs and structure its output.
system_instruction = """Determines whether the candidate is a Strong Fit, Partial Fit, or Not a Fit
Extracts the candidate's key skills
Identifies years of experience
Gives a short reason for the decision
Suggests whether the recruiter should shortlist the candidate
Returns the result as structured JSON
"""

## Define Response Schema

In [47]:
# This JSON schema ensures the Gemini model's output adheres to a specific structure and data types.
response_schema = {
    "type": "object",
    "properties": {
        "fit": {
            "type": "string"
        },
        "skills": {
            "type": "array",
            "items": {"type": "string"}
        },
        "experience_years": {
            "type": "integer"
        },
        "reason": {
            "type": "string"
        },
        "shortlist": {
            "type": "boolean"
        }
    },
    "required": [
        "fit",
        "skills",
        "experience_years",
        "reason",
        "shortlist"
    ]
}

## Helper Function to Analyze Resume

In [56]:
# This function encapsulates the Gemini API call for resume analysis.
def analyze_resume(job_description_input, resume_input):
  # The Gemini model will receive both the job description and resume as `contents`.
  # The `system_instruction` guides its behavior.
  response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=[job_description_input, resume_input],
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=response_schema,
        thinking_config=types.ThinkingConfig(
            thinking_level="low"
        ),
        max_output_tokens=500 # Ensure enough tokens for complete JSON output
    )
)
  return response.text

## Run Analysis and Display Results

In [57]:
# Call the helper function to analyze the resume.
result = analyze_resume(job_description, resume)

# Convert the JSON string output to a Python dictionary for easier access.
data = json.loads(result)

# Print the extracted information in a human-readable format.
print("--- Resume Analysis Results ---")
print(f"Fit: {data['fit']}")
print(f"Skills: {', '.join(data['skills'])}")
print(f"Years of Experience: {data['experience_years']}")
print(f"Reason: {data['reason']}")
print(f"Shortlist Candidate: {data['shortlist']}")

--- Resume Analysis Results ---
Fit: Strong Fit
Skills: Python, Machine Learning, SQL, NLP, AWS, Docker
Years of Experience: 4
Reason: Rahul meets all core requirements including Python, SQL, and 4 years of experience, and also possesses the preferred experience with NLP.
Shortlist Candidate: True
